# CardiaSense — DSP, augmentation & validation discipline

**Educational technical walkthrough · notebook 2 of 5**

**Objectives:** band-pass filtering, seeded augmentation, leakage-safe grouped CV splits

**You will need:** the `cardiasense` venv (run `./setup.sh`), the dataset at
`$CARDIASENSE_DATA_DIR` (downloaded by `setup.sh`), and a trained head for the
notebooks that use checkpoints (or run the smoke train in notebook 03 first).

> Educational / research prototype — **not a medical device**.

---




### Step 1 — Band-pass filtering (the task-dependent "lens")

Heart energy concentrates at **20–600 Hz**, lung sounds at **60–1500 Hz**
(Nyquist = 2000 Hz at 4 kHz). We apply a zero-phase Butterworth band-pass
(`scipy.signal.sosfiltfilt`) so the filter doesn't shift the signal in time.

**Technical note:** `sos` (second-order sections) + `sosfiltfilt` is numerically
stable and filters the signal *forwards then backwards*, removing phase lag —
essential for downstream time-aligned visualisation.


In [1]:

import os, sys
from pathlib import Path

ROOT = Path("/home/tnzr/CardiaSense-AI")
sys.path.insert(0, str(ROOT / "ml"))
sys.path.insert(0, str(ROOT))

from cardia.config import DEFAULT_DATA_DIR   # loads .env if present
DATA = os.environ.get("CARDIASENSE_DATA_DIR") or DEFAULT_DATA_DIR
print("Dataset root:", DATA)


Dataset root: /home/tnzr/CardiaSense-AI/data/HLS-CMDS


In [2]:

import numpy as np
import pandas as pd
from cardia.config import Config
from cardia.data.hls_cmds import build_task_manifest
from cardia.data.transforms import load_wav, peak_normalize, preprocess
from cardia.wandb_utils import viz
import matplotlib.pyplot as plt

cfg = Config()
lm = build_task_manifest(DATA, "lung", "binary")
row = lm.iloc[0]
raw, sr = load_wav(row["file_path"]); raw = peak_normalize(raw)
filt = preprocess(row["file_path"], cfg)          # lung band 60-1500 Hz
viz.overlay_fig(raw.numpy(), filt.numpy(), sr, f"band-pass overlay {row['sample_id']}")
plt.show()


/tmp/ipykernel_558896/3704940331.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



### Step 2 — Augmentation (train-only, and deterministic)

To generalise past 535 clips we synthesise plausible variants during training:
**time roll**, **amplitude scaling**, **additive Gaussian noise** on the
waveform, and **SpecAugment** masks on the spectrogram. Everything is driven by
a seeded `torch.Generator`, re-seeded per epoch, so runs are reproducible
(fold-0 reruns are bit-identical).

**Watch the difference** — the right panel shows the masked/perturbed version.


In [3]:

from cardia.data.transforms import Augment, MelSpec
aug = Augment(cfg, cfg.seed); aug.set_epoch(0)
w = aug.waveform(filt)
mel_spec = MelSpec(cfg.sample_rate, cfg.n_mels, cfg.n_fft, cfg.hop_length)
viz.aug_fig(mel_spec(filt).numpy(), aug.spec(mel_spec(w)).numpy(),
            f"augmentation {row['sample_id']}")
plt.show()


/tmp/ipykernel_558896/3539103049.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



### Step 3 — Leakage-safe, grouped, stratified splits

The **critical** methodological choice: a mixed session produces `H####`, `L####`
and `M####` clips that *must never* straddle train/val/test — otherwise the model
"recognises" the session instead of generalising.

The splitter:
1. **Outer** `StratifiedGroupKFold(5)` partitions **groups** (the session index)
   → fold *k* is TEST.
2. **Inner** re-split of the remaining groups → validation (~20%).
3. **Assertions** verify train/val/test group sets are pairwise disjoint and that
   the test folds cover every group exactly once.

Expected shape: ~60% train / 20% val / 20% test, with every fold containing all
classes (stratification).


In [4]:

from cardia.data.splits import make_splits
m = build_task_manifest(DATA, "sound", "binary")
splits = make_splits(m, n_folds=5, seed=0)   # asserts group disjointness
rows = [{"fold": s.fold, "train": len(s.train), "val": len(s.val), "test": len(s.test)} for s in splits]
print(pd.DataFrame(rows).to_string(index=False))


 fold  train  val  test
    0    321  107   107
    1    321  107   107
    2    321  107   107
    3    321  107   107
    4    321  107   107
